# Load dependencies

In [13]:
import os
from dotenv import load_dotenv
from supporting_functions import load_json, save_json, create_book_content_html_and_serve_with_flask, get_parsed_html_content, find_highlights_for_chapter
from html_functions import KindleHTMLParser, create_html

# Load all the files

In [14]:
book_name = "The Invisible Empire"
folder = "./books/invisible-empire/"
book_source = folder + "book.epub"
highlights_source = folder + "highlights.html"
highlights = KindleHTMLParser(highlights_source).highlights


In [15]:
from ebooklib import epub

book = epub.read_epub(book_source)

# Function to save content to a file
def save_file(file_path, content):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'wb') as f:
        f.write(content)

# Output directory
output_dir = folder + "unbundled_epub"

# Process each item in the EPUB
for item in book.get_items():
    file_path = os.path.join(output_dir, item.file_name)
    save_file(file_path, item.content)

print(f"EPUB unbundled successfully into {output_dir}")

d:\Projects\Fun\summarizer\.venv\Lib\site-packages\ebooklib\epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
d:\Projects\Fun\summarizer\.venv\Lib\site-packages\ebooklib\epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):


EPUB unbundled successfully into ./books/invisible-empire/unbundled_epub


In [43]:
#get the table of contents of the book
import ebooklib

def get_toc_details(book):
    toc = book.toc
    toc_details = []
    for item in toc:
        if isinstance(item, ebooklib.epub.Link):
            toc_item ={"type": "link", "href": item.href, "title": item.title, "uid": item.uid}

        elif isinstance(item, tuple) and isinstance(item[0], ebooklib.epub.Section):
            toc_item = {"type": "section", "title": item[0].title, "links": []}
            for link in item[1]:
                toc_item["links"].append({"type": "link", "href": link.href, "title": link.title, "uid": link.uid})
        toc_details.append(toc_item)
    return toc_details
toc = get_toc_details(book)
print(toc)


[{'type': 'link', 'href': 'xhtml/cover.xhtml', 'title': 'Cover', 'uid': 'cover'}, {'type': 'link', 'href': 'xhtml/toc.xhtml', 'title': 'Contents', 'uid': 'html-toc'}, {'type': 'link', 'href': 'xhtml/c001.xhtml', 'title': '1 BOUNTY', 'uid': 'c001'}, {'type': 'link', 'href': 'xhtml/c002.xhtml', 'title': '2 A WHOLE NEW WORLD', 'uid': 'c002'}, {'type': 'link', 'href': 'xhtml/c003.xhtml', 'title': '3 SUPERSIZE ME', 'uid': 'c003'}, {'type': 'link', 'href': 'xhtml/c004.xhtml', 'title': '4 THE VIRUS IS US', 'uid': 'c004'}, {'type': 'link', 'href': 'xhtml/c005.xhtml', 'title': '5 A DEEP CONTROL', 'uid': 'c005'}, {'type': 'link', 'href': 'xhtml/c006.xhtml', 'title': '6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS', 'uid': 'c006'}, {'type': 'link', 'href': 'xhtml/c007.xhtml', 'title': '7 A SPOTTY HISTORY OF THE SPECKLED MONSTER', 'uid': 'c007'}, {'type': 'link', 'href': 'xhtml/c008.xhtml', 'title': '8 GUT FEELING', 'uid': 'c008'}, {'type': 'link', 'href': 'xhtml/c009.xhtml', 'title': '9 A VIRUS VAN

In [ ]:
from bs4 import BeautifulSoup
def extract_clean_content_from_html(html_loc):
    with open(html_loc, 'rb') as f:
        content = f.read()
        if content:
            soup = BeautifulSoup(content.decode('utf-8'), 'html.parser')
            return soup.get_text()  
        return None

for item in toc:
    html_loc = folder + "unbundled_epub/" + item["href"]
    content = extract_clean_content_from_html(html_loc)



In [37]:
import os.path
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Document,
    load_index_from_storage,
    Settings
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-large"
)

Settings.llm = OpenAI(model="gpt-4o")

# check if storage already exists
PERSIST_DIR = "./storage"

In [45]:
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    repopulate = 'yes'  # Automatically repopulate if storage doesn't exist
else:
    # Ask the user if they want to repopulate the index
    repopulate = input("Storage already exists. Do you want to repopulate the index? (yes/no): ").strip().lower()

if repopulate == 'yes':
    # Load the documents and create a new index
    index = VectorStoreIndex([])
    for item in toc:
        html_loc = folder + "unbundled_epub/" + item["href"]
        content = extract_clean_content_from_html(html_loc)
        doc = Document(text=content, id_=item["href"], metadata={"href": item["href"], "title": item["title"]})
        index.insert(doc)
    # Store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # Load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

In [48]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

# Create a filter to match specific metadata
filters = MetadataFilters(filters=[ExactMatchFilter(key="href", value="xhtml/c001.xhtml")])

# Use the filter in the retriever
retriever = index.as_retriever(filters=filters)
response = retriever.retrieve("What does the chapter say?")
print(response)
